In [19]:
import requests
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from astropy.constants import G,c

In [20]:
#Define path for saving (one level up folder)
PLOT_DIR = "../plots"
DATA_DIR = "../data"
# Create the directory if it doesn't exist
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [21]:
def query_sdss(RA=None, DEC=None, radius_arcmin=None, save_filename="cluster_data.csv"):
    """
    Query SDSS DR16 SkyServer for galaxies around a given RA, DEC.

    Parameters
    ----------
    RA : float, optional
        Right Ascension (degrees). If None, user will be prompted.
    DEC : float, optional
        Declination (degrees). If None, user will be prompted.
    radius_arcmin : float, optional
        Search radius in arcminutes
    save_filename : str, optional. If None, user will be prompted
        CSV filename to save results (default = 'cluster_data.csv').

    Returns
    -------
    pd.DataFrame or None
        DataFrame with query results, or None if query fails.
    """

    # Prompt user if values not given
    if RA is None:
        RA = float(input("Enter RA (degrees): "))
    if DEC is None:
        DEC = float(input("Enter DEC (degrees): "))
    if radius_arcmin is None:
        radius_arcmin = float(input("Enter search radius (arcminutes): "))

    save_path = os.path.join(DATA_DIR, save_filename)
    query = f"""
    SELECT
        s.objid,
        sz.ra AS ra,
        sz.dec AS dec,
        pz.z AS photoz,
        pz.zerr AS photozerr,
        sz.z AS specz,
        sz.zerr AS speczerr,
        b.distance AS proj_sep,
        s.modelMag_u AS umag,
        s.modelMagErr_u AS umagerr,
        s.modelMag_g AS gmag,
        s.modelMagErr_g AS gmagerr,
        s.modelMag_r AS rmag,
        s.modelMagErr_r AS rmagerr,
        s.type AS obj_type
    FROM BESTDR16..PhotoObjAll AS s
    JOIN dbo.fGetNearbyObjEq({RA}, {DEC}, {radius_arcmin}) AS b
        ON b.objID = s.objID
    JOIN Photoz AS pz ON pz.objid = s.objid
    JOIN SpecObjAll AS sz ON sz.bestobjid = s.objid
    WHERE s.type = 3 AND sz.z > 0.05 AND sz.z < 0.20
    """

    url = "https://skyserver.sdss.org/dr16/en/tools/search/x_sql.aspx"
    params = {"cmd": query, "format": "csv"}

    print(f"📡 Querying SDSS SkyServer...\n   RA={RA}, DEC={DEC}, Radius={radius_arcmin} arcmin")
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"❌ Query failed: {e}")
        return None

    # Clean CSV (remove comment lines)
    lines = [line for line in response.text.splitlines() if not line.startswith('#')]
    clean_text = "\n".join(lines)

    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(clean_text)

    print(f"✅ Data saved to: {save_path}")

    # Load into DataFrame
    try:
        df = pd.read_csv(save_path)
        print(f" {len(df)} rows loaded into DataFrame")
        return df
    except Exception as e:
        print(f"!!! Failed to load DataFrame: {e}")
        return None

def save_plot(fig=None, filename="plot.png", save_dir="../plots", dpi=150):
    """
    Save a Matplotlib figure to a folder.

    Parameters
    ----------
    fig : matplotlib.figure.Figure, optional
        Figure to save. If None, uses current active figure.
    filename : str
        File name (including extension), e.g. "hist.png".
    save_dir : str
        Directory to save into (will be created if missing).
    dpi : int
        Resolution in dots per inch.
    """
    os.makedirs(save_dir, exist_ok=True)
    fig = fig or plt.gcf()
    path = os.path.join(save_dir, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    print(f"Plot saved to {path}")
    

In [22]:
df = query_sdss()
# Calculating the average specz for each id:
averaged_df = df.groupby('objid').agg({'specz': 'mean','ra': 'first','dec': 'first','proj_sep': 'first',}).reset_index()
averaged_df.describe()['specz']

📡 Querying SDSS SkyServer...
   RA=258.1294, DEC=64.0926, Radius=10.0 arcmin
✅ Data saved to: ../data\cluster_data.csv
 139 rows loaded into DataFrame


count    92.000000
mean      0.080838
std       0.008578
min       0.069976
25%       0.077224
50%       0.080961
75%       0.082797
max       0.150886
Name: specz, dtype: float64

In [23]:
# Compute mean and standard deviation for 'specz'
mean_specz = averaged_df['specz'].mean()
std_specz = averaged_df['specz'].std()

# Define 3-sigma limits
z_min = mean_specz - 3 * std_specz
z_max = mean_specz + 3 * std_specz

print(f"Mean spectroscopic redshift (specz): {mean_specz:.5f}")
print(f"Standard deviation of specz: {std_specz:.5f}")
print(f"3-sigma lower limit: {z_min:.5f}")
print(f"3-sigma upper limit: {z_max:.5f}")

Mean spectroscopic redshift (specz): 0.08084
Standard deviation of specz: 0.00858
3-sigma lower limit: 0.05510
3-sigma upper limit: 0.10657


In [24]:
# Histogram Plot
plt.figure(figsize=(10, 5))
plt.hist(averaged_df['specz'], bins=90, color='skyblue', edgecolor='black')
plt.title("Distribution of Spectroscopic Redshift for this Data")
plt.xlabel("Spectroscopic Redshift (specz)")
plt.ylabel("Number of Galaxies")
plt.grid(True, linestyle='--', alpha=0.7)
save_plot(filename="spectroscopic_redshift1_histogram.png")
plt.show()

Plot saved to ../plots\spectroscopic_redshift1_histogram.png


In [25]:
# Boxplot
plt.figure(figsize=(8, 2))
plt.boxplot(averaged_df['specz'], vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightgreen', color='black'),
            medianprops=dict(color='red'))
plt.title("Boxplot of Spectroscopic Redshift for this Data")
plt.xlabel("Spectroscopic Redshift (specz)")
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
save_plot(filename="spectroscopic_redshift1_boxplot.png")
plt.show()

Plot saved to ../plots\spectroscopic_redshift1_boxplot.png


In [26]:
# Filtering the data based on specz values, used 3 sigma deviation from mean as upper limit.
# Calculate mean and standard deviation of specz
mean_specz = averaged_df['specz'].mean()
std_specz = averaged_df['specz'].std()

# Define 3-sigma limits
z_min = mean_specz - 3 * std_specz
z_max = mean_specz + 3 * std_specz

print(f"Mean specz: {mean_specz:.5f}")
print(f"Standard Deviation: {std_specz:.5f}")
print(f"Redshift Range (3-sigma cut): [{z_min:.5f}, {z_max:.5f}]")

# Apply 3-sigma filter
filtered_df = averaged_df[(averaged_df['specz'] >= z_min) & (averaged_df['specz'] <= z_max)]

print(f"Number of galaxies before filtering: {len(averaged_df)}")
print(f"Number of galaxies after 3-sigma filtering: {len(filtered_df)}")

Mean specz: 0.08084
Standard Deviation: 0.00858
Redshift Range (3-sigma cut): [0.05510, 0.10657]
Number of galaxies before filtering: 92
Number of galaxies after 3-sigma filtering: 91


In [27]:
# Speed of light in km/s
c_km_s = c.to('km/s').value

# Calculate cluster mean redshift from filtered data
cluster_redshift = filtered_df['specz'].mean()

# Compute velocity for each galaxy using relativistic Doppler formula
filtered_df = df[(df['specz'] >= z_min) & (df['specz'] <= z_max)].copy()

filtered_df['velocity'] = c_km_s * ((1 + filtered_df['specz'])**2 - (1 + cluster_redshift)**2) / \
                                      ((1 + filtered_df['specz'])**2 + (1 + cluster_redshift)**2)

# Inspect new column
print(filtered_df[['specz', 'velocity']].head())

      specz    velocity
0  0.082447  659.731456
1  0.082466  664.999126
2  0.081218  319.185348
3  0.079561 -140.702385
4  0.079568 -138.855696


In [28]:
#plot the velocity column created as hist
plt.figure(figsize=(10, 5))
plt.hist(filtered_df['velocity'], bins=50, color='orchid', edgecolor='black')
plt.title("Velocity Distribution of Cluster Galaxies")
plt.xlabel("Velocity (km/s)")
plt.ylabel("Number of Galaxies")
plt.grid(True, linestyle='--', alpha=0.7)
save_plot(filename="velocity_distribution_galaxy_cluster.png")
plt.show()

Plot saved to ../plots\velocity_distribution_galaxy_cluster.png


In [29]:
# Step 1: Calculate the Mean Cluster Redshift
cluster_redshift = filtered_df['specz'].mean()
print(f"Mean Cluster Redshift (z_cluster): {cluster_redshift:.5f}")

# Step 2: Compute velocity of each galaxy relative to cluster redshift
filtered_df['velocity'] = c_km_s * ((1 + filtered_df['specz'])**2 - (1 + cluster_redshift)**2) / \
                                      ((1 + filtered_df['specz'])**2 + (1 + cluster_redshift)**2)

# Step 3: Calculate Velocity Dispersion (σ)
velocity_dispersion = filtered_df['velocity'].std()
print(f"Velocity Dispersion (σ): {velocity_dispersion:.2f} km/s")

Mean Cluster Redshift (z_cluster): 0.08003
Velocity Dispersion (σ): 1202.58 km/s


In [30]:
filtered_df['velocity'].describe()

count     137.000000
mean       -2.392825
std      1202.576295
min     -2803.471718
25%      -763.070775
50%       248.411265
75%       767.132350
max      4217.366753
Name: velocity, dtype: float64

In [31]:
print(f"The value of the cluster redshift = {cluster_redshift:.4f}")
print(f"The characteristic value of velocity dispersion of the cluster along the line of sight = {velocity_dispersion:.2f} km/s.")

The value of the cluster redshift = 0.0800
The characteristic value of velocity dispersion of the cluster along the line of sight = 1202.58 km/s.


In [32]:
#Plot histogram for proj sep column
plt.figure(figsize=(10,5))
plt.hist(filtered_df['proj_sep'], bins=30, color='cornflowerblue', edgecolor='black')
plt.title("Distribution of Projected Angular Separation of Cluster Galaxies")
plt.xlabel("Projected Separation (arcminutes)")
plt.ylabel("Number of Galaxies")
plt.grid(True, linestyle='--', alpha=0.7)
save_plot(filename="angular_speed_seperation.png")
plt.show()

Plot saved to ../plots\angular_speed_seperation.png


In [33]:
H_0 = 70 * 1000 / (3.086e22)   # Hubble constant in SI units (s^-1)
q0 = -0.534                    # Deceleration parameter
c_m_s = c.value                # Speed of light in m/s

# Cluster mean redshift
z_cluster = cluster_redshift

# Step 1: Co-moving distance (r) in meters
r = (c_m_s * z_cluster / H_0) * (1 - (z_cluster / 2) * (1 + q0))

# Step 2: Angular diameter distance (D_A) in meters
D_A = r / (1 + z_cluster)

# Step 3: Maximum angular separation (theta) in arcminutes
theta_arcmin = filtered_df['proj_sep'].max()  # in arcminutes

# Convert theta to radians
theta_rad = (theta_arcmin / 60) * (np.pi / 180)  # arcmin → degrees → radians

# Step 4: Physical diameter in meters
diameter = D_A * theta_rad

# Convert diameter to Megaparsecs (1 Mpc = 3.086e22 meters)
diameter_Mpc = diameter / 3.086e22

# Print results
print(f"r (co-moving distance) = {r:.3e} meters")
print(f"D_A (angular diameter distance) = {D_A:.3e} meters")
print(f"Diameter = {diameter:.3e} meters")
print(f"Diameter = {diameter_Mpc:.3f} Mpc")

r (co-moving distance) = 1.038e+25 meters
D_A (angular diameter distance) = 9.611e+24 meters
Diameter = 2.752e+22 meters
Diameter = 0.892 Mpc


In [34]:
# Gravitational Constant in SI units (m^3 kg^-1 s^-2)
G_si = G.value

# Step 1: Velocity dispersion (sigma) in m/s
sigma_m_s = velocity_dispersion * 1000  # convert km/s to m/s

# Step 2: Cluster radius in meters (half the physical diameter)
R = diameter / 2

# Step 3: Dynamical mass in kg
M_dyn_kg = (3 * sigma_m_s**2 * R) / G_si

# Step 4: Convert to solar masses
M_dyn_solar = M_dyn_kg / 1.989e30

# Print results
print(f"Dynamical Mass of the Cluster = {M_dyn_kg:.3e} kg")
print(f"Dynamical Mass of the Cluster = {M_dyn_solar:.3e} solar masses")

Dynamical Mass of the Cluster = 8.945e+44 kg
Dynamical Mass of the Cluster = 4.497e+14 solar masses


In [35]:
# Number of galaxies (filtered cluster members)
N_galaxies = len(filtered_df)

# Assumed stellar mass of one galaxy (in solar masses)
M_galaxy_stellar = 1e11  # solar masses

# Total luminous (stellar) mass of the cluster
M_luminous = N_galaxies * M_galaxy_stellar

print(f"Estimated total luminous (stellar) mass = {M_luminous:.2e} solar masses")
print(f"Estimated total dynamical mass = {M_dyn_solar:.2e} solar masses")

# Ratio: how much more is the dynamical mass?
dark_matter_factor = M_dyn_solar / M_luminous
print(f"The dynamical mass is {dark_matter_factor:.2f} times the luminous mass.")

Estimated total luminous (stellar) mass = 1.37e+13 solar masses
Estimated total dynamical mass = 4.50e+14 solar masses
The dynamical mass is 32.83 times the luminous mass.


In [36]:
# Masses in solar masses
masses = [M_luminous, M_dyn_solar]
labels = ['Luminous Mass (Stars)', 'Dynamical Mass (Total)']

plt.figure(figsize=(8,5))
plt.bar(labels, masses, color=['skyblue', 'orange'], edgecolor='black')
plt.yscale('log')  # Because the dynamical mass is much larger
plt.ylabel("Mass (Solar Masses)")
plt.title("Comparison of Luminous Mass vs Dynamical Mass of the Galaxy Cluster")
plt.grid(True, which="both", ls="--", linewidth=0.5, alpha=0.7)
save_plot(filename="luminous_dynamical_mass_comparison.png")
plt.show()

Plot saved to ../plots\luminous_dynamical_mass_comparison.png
